# Stable Cascade: Cascaded Latent Diffusion

3-stage cascaded latent diffusion. Instead of 8x compression (SD 1.5), Stable Cascade uses **42x compression** in Stage C for the main generation, then Stage B and Stage A upsample/refine.

**Key insight:** Doing diffusion in a highly compressed space is much cheaper, and the cascade recovers quality.

- **Stage C (Prior):** Text-conditioned diffusion in a tiny 24x24 latent space. This is where all the semantic work happens.
- **Stage B:** Upsamples Stage C's output to a higher-res latent
- **Stage A (VQGAN):** Decodes to full pixel resolution (1024x1024)

Pernias et al. ([2402.01355](https://arxiv.org/abs/2402.01355)).

**Time:** ~2 hrs | **Hardware:** ~8 GB VRAM (CPU offloading available)

In [ ]:
# GPU: ~8 GB VRAM (with CPU offloading)
import torch
import gc
from diffusers import StableCascadeDecoderPipeline, StableCascadePriorPipeline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Stage C (Prior) -- operates in highly compressed 24x24 space
print("\nLoading Stage C (Prior)...")
prior = StableCascadePriorPipeline.from_pretrained(
    "stabilityai/stable-cascade-prior",
    torch_dtype=dtype,
)
prior.enable_model_cpu_offload()
print("Stage C loaded.")

# Stage B + A (Decoder) -- upsamples to full resolution
print("Loading Stage B+A (Decoder)...")
decoder = StableCascadeDecoderPipeline.from_pretrained(
    "stabilityai/stable-cascade",
    torch_dtype=dtype,
)
decoder.enable_model_cpu_offload()
print("Stage B+A loaded.")

In [ ]:
# EXERCISE: Compute and compare VAE compression ratios

def compute_compression_ratio(
    input_shape: tuple[int, ...],
    latent_shape: tuple[int, ...],
) -> dict:
    """Compute spatial and total compression ratios.

    Args:
        input_shape: (C, H, W) of pixel-space image
        latent_shape: (C, H, W) of latent representation

    Returns:
        dict with: spatial_ratio, channel_ratio, total_ratio,
        input_elements, latent_elements
    """
    # YOUR CODE
    raise NotImplementedError


# Compare SD 1.5 vs Stable Cascade
sd15 = compute_compression_ratio((3, 512, 512), (4, 64, 64))
cascade = compute_compression_ratio((3, 1024, 1024), (16, 24, 24))
print(f"SD 1.5:  {sd15['total_ratio']:.1f}x compression")
print(f"Cascade: {cascade['total_ratio']:.1f}x compression")


In [ ]:
# Inspect the 3 stages: architecture and param counts

def count_params(model):
    """Count parameters in millions."""
    return sum(p.numel() for p in model.parameters()) / 1e6

def print_stage_summary(name, model, depth=1):
    """Print architecture summary for a pipeline component."""
    print(f"\n{'=' * 70}")
    print(f"  {name}: {count_params(model):.1f}M parameters")
    print(f"{'=' * 70}")
    for child_name, child in model.named_children():
        child_params = count_params(child)
        if child_params > 0.1:  # Only show non-trivial components
            print(f"  ├── {child_name}: {type(child).__name__} ({child_params:.1f}M)")
            if depth > 0:
                for sub_name, sub_child in child.named_children():
                    sub_params = count_params(sub_child)
                    if sub_params > 0.1:
                        print(f"  │   ├── {sub_name}: {type(sub_child).__name__} ({sub_params:.1f}M)")

# Stage C: Prior (the main generation model)
print_stage_summary("Stage C (Prior)", prior.prior)

# Stage B: Decoder backbone
print_stage_summary("Stage B (Decoder backbone)", decoder.decoder)

# Stage A: VQGAN
if hasattr(decoder, 'vqgan'):
    print_stage_summary("Stage A (VQGAN)", decoder.vqgan)

# Summary comparison
print(f"\n{'=' * 70}")
print(f"  COMPRESSION COMPARISON")
print(f"{'=' * 70}")
print(f"  SD 1.5:           512x512 image → 64x64x4 latent  (8x spatial compression)")
print(f"  Stable Cascade:   1024x1024 image → 24x24xC latent (42x spatial compression!)")
print(f"")
print(f"  SD 1.5 latent elements:    64 * 64 * 4 = {64*64*4:,}")
print(f"  SC Stage C latent elements: 24 * 24 * 16 = {24*24*16:,}  (estimated C=16)")
print(f"")
print(f"  Key: SC does diffusion on {24*24*16:,} values instead of {64*64*4:,}")
print(f"  That's {64*64*4 / (24*24*16):.1f}x fewer latent elements -- much faster training & inference")

In [ ]:
# GPU: ~8 GB VRAM with CPU offloading
# Generate an image through the full 3-stage cascade

prompt = "a futuristic city at sunset, highly detailed, cinematic lighting"
negative_prompt = "blurry, low quality, distorted"

print(f"Prompt: '{prompt}'")
print(f"\n--- Stage C: Generating compressed latent (24x24) ---")

# Stage C: generate highly compressed latent
prior_output = prior(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=20,
    guidance_scale=4.0,
)

# Capture Stage C output for visualization
stage_c_embeddings = prior_output.image_embeddings
print(f"Stage C output shape: {stage_c_embeddings.shape}")
print(f"Stage C output dtype: {stage_c_embeddings.dtype}")

print(f"\n--- Stage B+A: Decoding to full resolution ---")

# Stage B + A: decode to full resolution
output = decoder(
    image_embeddings=stage_c_embeddings,
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=10,
    guidance_scale=0.0,  # No guidance needed for decoder
)

final_image = output.images[0]
print(f"Final image size: {final_image.size}")

# Display
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(final_image)
ax.set_title(f"Stable Cascade Output\n\"{prompt}\"", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Visualize intermediate stage outputs
# Show the progression: Stage C latent → Stage B → Final output

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Stage C latent visualization (24x24 -- highly compressed)
stage_c_np = stage_c_embeddings[0].float().cpu().numpy()
# Show first few channels as RGB-ish composite
n_channels = stage_c_np.shape[0]
print(f"Stage C latent: {stage_c_np.shape} (channels x H x W)")

# Take first 3 channels for visualization (or fewer if not enough)
vis_channels = min(3, n_channels)
stage_c_vis = stage_c_np[:vis_channels]
# Normalize each channel independently to [0, 1]
for i in range(vis_channels):
    ch = stage_c_vis[i]
    stage_c_vis[i] = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)

if vis_channels == 3:
    axes[0].imshow(np.transpose(stage_c_vis, (1, 2, 0)))
else:
    axes[0].imshow(stage_c_vis[0], cmap='viridis')
    
spatial_h, spatial_w = stage_c_np.shape[1], stage_c_np.shape[2]
axes[0].set_title(
    f"Stage C Latent\n{spatial_h}x{spatial_w}x{n_channels}\n(42x compression)",
    fontsize=12
)
axes[0].axis("off")

# 2. Channel-wise heatmaps of Stage C
# Show a grid of individual channels
n_show = min(16, n_channels)
grid_size = int(np.ceil(np.sqrt(n_show)))
channel_grid = np.zeros((grid_size * spatial_h, grid_size * spatial_w))
for idx in range(n_show):
    r, c = idx // grid_size, idx % grid_size
    ch = stage_c_np[idx]
    ch = (ch - ch.min()) / (ch.max() - ch.min() + 1e-8)
    channel_grid[r * spatial_h:(r + 1) * spatial_h, c * spatial_w:(c + 1) * spatial_w] = ch

axes[1].imshow(channel_grid, cmap='viridis')
axes[1].set_title(
    f"Stage C Channels (first {n_show} of {n_channels})\nEach {spatial_h}x{spatial_w}",
    fontsize=12
)
axes[1].axis("off")

# 3. Final output
axes[2].imshow(final_image)
w, h = final_image.size
axes[2].set_title(f"Final Output\n{w}x{h}\n(after Stage B + A)", fontsize=12)
axes[2].axis("off")

plt.suptitle("Stable Cascade: From Compressed Latent to Full Resolution", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nCompression path:")
print(f"  {w}x{h} pixels → {spatial_h}x{spatial_w}x{n_channels} latent → {w}x{h} pixels")
print(f"  Spatial compression: {w // spatial_h}x ({w}→{spatial_h})")

In [ ]:
# Compression comparison: SD 1.5 vs Stable Cascade
# Visualize relative sizes and compute exact ratios

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- Left: Visual comparison of latent sizes ---
ax = axes[0]
ax.set_xlim(0, 110)
ax.set_ylim(0, 110)
ax.set_aspect('equal')

# SD 1.5: 512x512 image, 64x64x4 latent
# Stable Cascade: 1024x1024 image, 24x24x16 latent
# Scale everything to fit: 1024px = 100 units

# SC image (1024x1024) -> 100x100
rect_sc_img = mpatches.Rectangle((5, 5), 100, 100, linewidth=2,
                                  edgecolor='#2ecc71', facecolor='#2ecc71', alpha=0.15,
                                  label='SC image 1024x1024')
ax.add_patch(rect_sc_img)
ax.text(55, 108, "SC: 1024x1024", ha='center', fontsize=10, color='#2ecc71', fontweight='bold')

# SD image (512x512) -> 50x50
rect_sd_img = mpatches.Rectangle((5, 5), 50, 50, linewidth=2,
                                  edgecolor='#3498db', facecolor='#3498db', alpha=0.15,
                                  label='SD image 512x512')
ax.add_patch(rect_sd_img)
ax.text(30, 57, "SD: 512x512", ha='center', fontsize=10, color='#3498db', fontweight='bold')

# SD latent (64x64) -> 6.25x6.25
rect_sd_lat = mpatches.Rectangle((5, 5), 6.25, 6.25, linewidth=2,
                                  edgecolor='#3498db', facecolor='#3498db', alpha=0.6)
ax.add_patch(rect_sd_lat)
ax.text(8.5, 13, "SD latent\n64x64", ha='center', fontsize=8, color='#3498db')

# SC latent (24x24) -> 2.34x2.34
rect_sc_lat = mpatches.Rectangle((5, 5), 2.34, 2.34, linewidth=2,
                                  edgecolor='#e74c3c', facecolor='#e74c3c', alpha=0.8)
ax.add_patch(rect_sc_lat)
ax.text(6.2, 9, "SC latent\n24x24", ha='center', fontsize=7, color='#e74c3c')

ax.set_title("Relative Spatial Sizes\n(image vs latent space)", fontsize=13)
ax.axis("off")

# --- Right: Detailed numbers ---
ax2 = axes[1]

comparison_data = [
    ["Metric", "SD 1.5", "Stable Cascade"],
    ["Output resolution", "512x512", "1024x1024"],
    ["Latent spatial", "64x64", "24x24"],
    ["Latent channels", "4", "16 (est.)"],
    ["Spatial compression", "8x", "42x"],
    ["Total latent elements", f"{64*64*4:,}", f"{24*24*16:,}"],
    ["Pixels per latent", f"{512*512 / (64*64*4):.0f}", f"{1024*1024 / (24*24*16):.0f}"],
    ["Diffusion cost (rel.)", "1.0x", f"{24*24*16 / (64*64*4):.2f}x"],
    ["Training efficiency", "Baseline", "~16x faster (paper)"],
]

table = ax2.table(
    cellText=comparison_data[1:],
    colLabels=comparison_data[0],
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# Color header
for j in range(3):
    table[0, j].set_facecolor('#34495e')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Highlight key rows
for row in [3, 4, 6, 7]:  # spatial compression, total elements, cost, efficiency
    table[row, 2].set_facecolor('#d5f5e3')

ax2.axis("off")
ax2.set_title("Compression Numbers", fontsize=13)

plt.tight_layout()
plt.show()

print("\nThe 42x spatial compression is the key innovation.")
print("Diffusion operates on far fewer elements → dramatically cheaper.")
print("The cascade (B → A) recovers visual quality from the compressed representation.")

In [ ]:
# GPU: ~10 GB VRAM total (runs sequentially with CPU offload)
# Quality comparison: SD 1.5 vs Stable Cascade on same prompt

from diffusers import StableDiffusionPipeline
import time

comparison_prompt = "a detailed painting of a medieval castle on a cliff, dramatic sky, golden hour"

# --- Stable Cascade ---
print("Generating with Stable Cascade...")
torch.manual_seed(42)
t_start = time.time()

prior_output_cmp = prior(
    prompt=comparison_prompt,
    num_inference_steps=20,
    guidance_scale=4.0,
)
sc_images = decoder(
    image_embeddings=prior_output_cmp.image_embeddings,
    prompt=comparison_prompt,
    num_inference_steps=10,
    guidance_scale=0.0,
).images

sc_time = time.time() - t_start
sc_img = sc_images[0]
print(f"  Stable Cascade: {sc_time:.1f}s, {sc_img.size[0]}x{sc_img.size[1]}")

# Free VRAM for SD 1.5
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

# --- SD 1.5 ---
print("Loading SD 1.5...")
try:
    sd_pipe = StableDiffusionPipeline.from_pretrained(
        "stable-diffusion-v1-5/stable-diffusion-v1-5",
        torch_dtype=dtype,
    )
    sd_pipe.enable_model_cpu_offload()

    print("Generating with SD 1.5...")
    torch.manual_seed(42)
    t_start = time.time()

    sd_images = sd_pipe(
        prompt=comparison_prompt,
        num_inference_steps=30,
        guidance_scale=7.5,
    ).images

    sd_time = time.time() - t_start
    sd_img = sd_images[0]
    print(f"  SD 1.5: {sd_time:.1f}s, {sd_img.size[0]}x{sd_img.size[1]}")
    sd_loaded = True

    del sd_pipe
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

except Exception as e:
    print(f"  Could not load SD 1.5: {e}")
    print("  Skipping direct comparison.")
    sd_loaded = False

# Display comparison
if sd_loaded:
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(sd_img)
    axes[0].set_title(
        f"SD 1.5\n{sd_img.size[0]}x{sd_img.size[1]} | {sd_time:.1f}s | 30 steps\n"
        f"64x64x4 latent (8x compression)",
        fontsize=11
    )
    axes[0].axis("off")

    axes[1].imshow(sc_img)
    axes[1].set_title(
        f"Stable Cascade\n{sc_img.size[0]}x{sc_img.size[1]} | {sc_time:.1f}s | 20+10 steps\n"
        f"24x24 latent (42x compression)",
        fontsize=11
    )
    axes[1].axis("off")

    fig.suptitle(f'"{comparison_prompt}"', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

    print(f"\nSD 1.5:          {sd_time:.1f}s at {sd_img.size[0]}x{sd_img.size[1]}")
    print(f"Stable Cascade:  {sc_time:.1f}s at {sc_img.size[0]}x{sc_img.size[1]} (4x more pixels)")
else:
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(sc_img)
    ax.set_title(f"Stable Cascade\n{sc_img.size[0]}x{sc_img.size[1]} | {sc_time:.1f}s", fontsize=12)
    ax.axis("off")
    plt.show()

## Checkpoint

**Stable Cascade in 2 sentences:**

> "Stable Cascade does diffusion in a 42x-compressed latent space using a 3-stage cascade -- Stage C generates the semantic structure cheaply, then B and A reconstruct the visual details. The extreme compression makes training significantly more efficient while the cascade recovers the quality that compression loses."

**Key contributions:**
1. **Much higher compression ratio** than prior work (42x vs SD's 8x) -- enables diffusion on ~9K elements instead of ~16K, despite generating 4x more pixels
2. **Cascaded refinement** recovers quality from aggressive compression -- Stage B adds spatial detail, Stage A decodes to pixels
3. **Faster training and inference** -- diffusion in the tiny 24x24 space is dramatically cheaper per step

**Architecture lineage:**
- **Würstchen** (2306.00637) -- precursor to Stable Cascade, proved the cascaded approach works
- **Stable Cascade** (2402.01355) -- scaled up with better Stage C architecture, released by Stability AI

**Further reading:**
- Stable Cascade: [2402.01355](https://arxiv.org/abs/2402.01355)
- Würstchen: [2306.00637](https://arxiv.org/abs/2306.00637)